# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane (confirmed, locking this week):** Lane 2 — Refresh / Content Opportunity Scoring.
The question stays the same as weeks 1–3: *which pages should a reviewer look at first?*

**Data for this notebook:** the 30k-row starter slice `data/raw/content_refresh_anonymized.csv`
(ships in the repo, so this notebook runs top to bottom anywhere, including Colab, with no token).
The warehouse version of this baseline is the Week-5 step.

**Evaluation label (never an input):** `is_declining = (trend_direction == "down")` — the same proxy as Week 2.
It is used ONLY to grade the queue at the end of section 2.

In [1]:
import os, sys, json, subprocess
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/susheel123-sketch/Flyrank-Internship-ml"
REPO_DIR = "Flyrank-Internship-ml"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows, cols:", df.shape, "| clients:", df["client_id"].nunique())

# Label for EVALUATION ONLY. Kept in a separate Series, never inside the rule.
y = (df["trend_direction"] == "down").astype(int)
print(f"base rate (share of pages declining): {y.mean():.3f}  (n={len(y)})")

# Columns the rule is NOT allowed to read: label source + the two windows that build the label.
FORBIDDEN = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
             "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]

rows, cols: (30000, 44) | clients: 32
base rate (share of pages declining): 0.542  (n=30000)


## 1. My rule, the two signals it leans on, and its reason code

**My first idea (before checking):** "Refresh pages that are *stale* and *visible*, and fix pages whose
CTR is too low for their position." That leans on two signals from the session's real FlyRank flags:

| Signal | FlyRank flag it sits behind | What I expect |
|---|---|---|
| A. Staleness (`days_since_last_update`) | the refresh flags (`stale_visible_page`, ≥180 days) | older last-update → more decline |
| B. CTR vs position (`ctr` against the normal CTR for that position band) | the CTR-fix logic (`low_ctr_visible_page`) | CTR far below its band → more decline |

Each check below prints a bucket table **with n**, then a harder test: does the gap survive when I hold
traffic level fixed, and does it hold *inside* clients (so it is not just one client's habit)?

### Signal A — staleness

In [2]:
t = df.assign(y=y)
stale_b = pd.cut(t["days_since_last_update"], [-1, 30, 90, 120, 180, 9999],
                 labels=["0-30", "31-90", "91-120", "121-180", "181+"])
tab_a = t.groupby(stale_b, observed=True).agg(n=("y", "size"), decline_rate=("y", "mean"),
                                               median_impressions=("impressions_90d", "median")).round(3)
print("A1. Decline rate by days since last update (all pages)")
display(tab_a)

# A2: hold traffic level fixed (visible pages only, >=500 impressions)
vis = t[t["impressions_90d"] >= 500].copy()
vis["stale_90"] = np.where(vis["days_since_last_update"] > 90, "updated >90d ago", "updated <=90d ago")
vis["traffic"] = pd.cut(vis["impressions_90d"], [500, 1000, 3000, 30000, 1e9],
                        labels=["500-1k", "1k-3k", "3k-30k", "30k+"])
tab_a2 = vis.groupby(["traffic", "stale_90"], observed=True)["y"].agg(n="size", decline_rate="mean").round(3).unstack()
print("A2. Same split, inside traffic tiers (visible pages, n shown)")
display(tab_a2)

# A3: inside clients — clients with >=30 pages on each side
g = vis.groupby(["client_id", "stale_90"])["y"].agg(["size", "mean"]).unstack().dropna()
g = g[(g[("size", "updated >90d ago")] >= 30) & (g[("size", "updated <=90d ago")] >= 30)]
diff_a = g[("mean", "updated >90d ago")] - g[("mean", "updated <=90d ago")]
print(f"A3. Inside clients: {len(g)} clients have both groups; median gap = {diff_a.median():+.3f}; "
      f"stale side higher in {(diff_a > 0).sum()} of {len(g)}")
print("Who sits in the 91-120 bucket:", t.loc[stale_b == "91-120", "client_id"].nunique(), "clients;",
      f"top 2 clients hold {t.loc[stale_b == '91-120', 'client_id'].value_counts().head(2).sum() / (stale_b == '91-120').sum():.0%} of it")
print("Pages the session's 180-day threshold can even fire on:", int((t['days_since_last_update'] >= 180).sum()),
      "| of those with >=500 impressions:", int(((t['days_since_last_update'] >= 180) & (t['impressions_90d'] >= 500)).sum()))

A1. Decline rate by days since last update (all pages)


,n,decline_rate,median_impressions
days_since_last_update,,,
0-30,20480,0.511,470.0
31-90,175,0.589,510.0
91-120,9115,0.613,1721.0
121-180,56,0.286,4.0
181+,174,0.471,15.5


A2. Same split, inside traffic tiers (visible pages, n shown)


n                       decline_rate                 
stale_90 updated <=90d ago updated >90d ago updated <=90d ago updated >90d ago
traffic                                                                       
500-1k                2113             1093             0.574            0.651
1k-3k                 3271             1955             0.612            0.670
3k-30k                4170             3035             0.577            0.599
30k+                   589              489             0.484            0.436

A3. Inside clients: 7 clients have both groups; median gap = +0.005; stale side higher in 4 of 7
Who sits in the 91-120 bucket: 18 clients; top 2 clients hold 66% of it
Pages the session's 180-day threshold can even fire on: 174 | of those with >=500 impressions: 17


**Verdict A — staleness: MIXED.**

- The raw table looks like a win: 91–120 days → 61% decline vs 51% for 0–30 days.
- But it falls apart under pressure:
  - Updates are not spread out. They come in **batches**: almost every page is either 0–30 or 91–120 days old.
    The 91–120 bucket is mostly two clients. So "stale" here mostly means "belongs to a client who updated in one batch long ago".
  - Inside clients the gap is close to **zero** (see A3). The pooled gap is a client effect, not a page effect.
  - For the biggest pages (30k+ impressions) it flips sign.
  - The session's ≥180-day threshold fires on under 200 pages, and only a handful have real traffic. On this slice it is close to a dead rule.
- **What this saved:** my first rule idea (stale × visible) would have been ranking clients, not pages. I dropped staleness from the score.

### Signal B — CTR vs position

In [3]:
# Only pages where CTR is readable: real position data (avg_position > 0, since 0 = "no data")
# and a volume floor of 500 impressions (below that one click swings CTR a lot).
pos_band = pd.cut(df["avg_position"].where(df["avg_position"] > 0), [0, 3, 10, 20, 50, 999],
                  labels=["1-3", "4-10", "11-20", "21-50", "50+"])
readable = (df["impressions_90d"] >= 500) & pos_band.notna()
band_ctr = df[readable].groupby(pos_band[readable], observed=True)["ctr"].median()
print("Normal (median) CTR per position band, readable pages — CTR is x100, so 0.24 means 0.24%")
display(band_ctr.to_frame("median_ctr_pct").assign(n=df[readable].groupby(pos_band[readable], observed=True).size()))

b = df[readable].assign(y=y[readable], band=pos_band[readable])
b["ctr_vs_band"] = b["ctr"] / b["band"].map(band_ctr).astype(float).replace(0, np.nan)
b["ctr_bucket"] = pd.cut(b["ctr_vs_band"], [-0.01, 0.25, 0.5, 1, 2, 1e9],
                         labels=["<25% of band", "25-50%", "50-100%", "1-2x", ">2x"])
tab_b = b.groupby("ctr_bucket", observed=True).agg(n=("y", "size"), decline_rate=("y", "mean"),
                                                   median_impressions=("impressions_90d", "median")).round(3)
print("B1. Decline rate by CTR relative to its position band")
display(tab_b)

b["low_ctr"] = np.where(b["ctr_vs_band"] < 0.5, "CTR < half of band", "CTR normal or better")
tab_b2 = b.groupby(["band", "low_ctr"], observed=True)["y"].agg(n="size", decline_rate="mean").round(3).unstack()
print("B2. Same split inside each position band (n shown)")
display(tab_b2)

g = b.groupby(["client_id", "low_ctr"])["y"].agg(["size", "mean"]).unstack().dropna()
g = g[(g[("size", "CTR < half of band")] >= 30) & (g[("size", "CTR normal or better")] >= 30)]
diff_b = g[("mean", "CTR < half of band")] - g[("mean", "CTR normal or better")]
print(f"B3. Inside clients: {len(g)} clients have both groups; median gap = {diff_b.median():+.3f}; "
      f"low-CTR side higher in {(diff_b > 0).sum()} of {len(g)}")

Normal (median) CTR per position band, readable pages — CTR is x100, so 0.24 means 0.24%


,median_ctr_pct,n
avg_position,,
1-3,0.20,480
4-10,0.24,7084
11-20,0.17,4459
21-50,0.09,4314
50+,0.00,389


B1. Decline rate by CTR relative to its position band


,n,decline_rate,median_impressions
ctr_bucket,,,
<25% of band,3033,0.682,1239.0
25-50%,1688,0.675,4199.0
50-100%,3636,0.618,3508.0
1-2x,4070,0.582,3508.5
>2x,3910,0.517,4104.5


B2. Same split inside each position band (n shown)


n                            decline_rate                     
low_ctr CTR < half of band CTR normal or better CTR < half of band CTR normal or better
band                                                                                   
1-3                  141.0                339.0              0.887                0.676
4-10                1686.0               5398.0              0.706                0.539
11-20               1206.0               3253.0              0.699                0.584
21-50               1500.0               2814.0              0.617                0.613
50+                    NaN                389.0                NaN                0.298

B3. Inside clients: 16 clients have both groups; median gap = +0.121; low-CTR side higher in 15 of 16


**Verdict B — CTR vs position: CONFIRMED (for positions 1–20).**

- Clean, step-by-step pattern: the further CTR falls below its position band, the higher the decline rate (B1).
- It holds inside each band from 1 to 20, and the gap is biggest at the top of page 1 (B2).
- It holds **inside clients** (B3) — almost every client shows it. So it is a page-level signal, not a client habit.
- It does **not** hold at positions 21–50 (gap ≈ 0) or 50+ (nobody clicks there anyway). So the rule only fires on positions 1–20.
- Honest limit: CTR and the label come from the same 90-day snapshot. This is an **observed link**, not proof that low CTR *causes* decline.
  One harmless story fits too: a page that briefly picked up a burst of loosely related searches gets a low CTR, and those impressions then fade.

### The rule (plain words)

> **If a page sits on positions 1–20, had at least 500 impressions in 90 days, and its CTR is less than half the
> normal CTR for its position band, flag it for a title/meta rewrite. Rank flagged pages by how far below normal
> the CTR is, scaled by how many people saw the page (log of impressions).**

- **Score** = `ctr_gap × log(1 + impressions_90d)`, where `ctr_gap = 1 − ctr / band_ctr` (0 to 1). Unflagged pages score 0.
- **Reason code (one):** `ctr_below_position_norm`
- **Action label:** `rewrite_title_meta` (flagged) / `monitor` (not flagged)
- No fitted weights. The only numbers that come from the data are the 3 band medians printed above — computed without the label.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`. Then grade it once with precision@K against the base rate.*

In [4]:
RULE_BANDS = ["1-3", "4-10", "11-20"]          # positions 1-20 only (signal B is flat beyond 20)
MIN_IMPRESSIONS = 500
LOW_CTR_SHARE = 0.5

q = df.copy()
q["position_band"] = pos_band.astype(object)
q["band_ctr"] = q["position_band"].map(band_ctr).astype(float)
eligible = (q["impressions_90d"] >= MIN_IMPRESSIONS) & q["position_band"].isin(RULE_BANDS)
q["ctr_gap"] = (1 - q["ctr"] / q["band_ctr"]).clip(0, 1).where(eligible, 0.0)
q["flagged"] = eligible & (q["ctr"] < LOW_CTR_SHARE * q["band_ctr"])
q["baseline_score"] = np.where(q["flagged"], q["ctr_gap"] * np.log1p(q["impressions_90d"]), 0.0).round(4)
q["reason_code"] = np.where(q["flagged"], "ctr_below_position_norm", "no_flag")
q["action"] = np.where(q["flagged"], "rewrite_title_meta", "monitor")
q = q.sort_values(["baseline_score", "impressions_90d"], ascending=False).reset_index(drop=True)
q["rank"] = np.arange(1, len(q) + 1)

OUT_COLS = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action",
            "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_band", "band_ctr", "ctr_gap",
            "content_type", "main_intent", "days_since_last_update", "content_age_days"]
os.makedirs("work/outputs", exist_ok=True)
q[OUT_COLS].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv:", len(q), "rows |",
      int(q["flagged"].sum()), f"flagged ({q['flagged'].mean():.1%})")

# Leakage guard: the rule must not have read any forbidden column.
rule_inputs = ["impressions_90d", "ctr", "avg_position"]
assert not set(rule_inputs) & set(FORBIDDEN) and not set(OUT_COLS) & set(FORBIDDEN)
print("rule inputs:", rule_inputs, "| forbidden columns used: none")

wrote work/outputs/baseline_action_score.csv: 30000 rows | 3033 flagged (10.1%)
rule inputs: ['impressions_90d', 'ctr', 'avg_position'] | forbidden columns used: none


In [5]:
# Grade the queue ONCE. Label joined back by content_id; it never touched the score.
lab = pd.Series(y.values, index=df["content_id"])
q["is_declining"] = q["content_id"].map(lab).values
base = q["is_declining"].mean()
ks = [10, 20, 50, 100, 200, 500]
rows = [{"K": k, "precision_at_K": round(q.head(k)["is_declining"].mean(), 3),
         "base_rate": round(base, 3), "lift": round(q.head(k)["is_declining"].mean() / base, 2),
         "clients_in_topK": q.head(k)["client_id"].nunique()} for k in ks]
patk = pd.DataFrame(rows)
display(patk)
print(f"precision over ALL {int(q['flagged'].sum())} flagged pages: {q.loc[q['flagged'], 'is_declining'].mean():.3f}  vs base {base:.3f}")
print("dummy baseline (random order) = base rate =", round(base, 3))

,K,precision_at_K,base_rate,lift,clients_in_topK
0,10,0.600,0.542,1.11,5
1,20,0.600,0.542,1.11,7
2,50,0.640,0.542,1.18,11
3,100,0.690,0.542,1.27,14
4,200,0.700,0.542,1.29,16
5,500,0.708,0.542,1.31,20


precision over ALL 3033 flagged pages: 0.712  vs base 0.542
dummy baseline (random order) = base rate = 0.542


**Sensitivity check (not tuning):** I want to know how much the *ranking formula* matters, separate from
the *flag*. Below, the same flagged set is ranked three ways. I keep `gap × log(impressions)` because it is the
version I can defend in words (how wrong the CTR is, weighted by how many people saw it). I am **not**
switching to whichever row scores highest — that would be fitting the baseline to the label.

In [6]:
f = q[q["flagged"]].copy()
variants = {
    "gap x log(impressions)  [CHOSEN]": f["ctr_gap"] * np.log1p(f["impressions_90d"]),
    "missed clicks = impressions x (band_ctr - ctr)": f["impressions_90d"] * (f["band_ctr"] - f["ctr"]) / 100,
    "gap only (impressions as tie-break)": f["ctr_gap"] + np.log1p(f["impressions_90d"]) / 1000,
}
sens = pd.DataFrame({name: [f.loc[s.sort_values(ascending=False).index[:k], "is_declining"].mean().round(3) for k in ks]
                     for name, s in variants.items()}, index=[f"P@{k}" for k in ks])
display(sens)
print("Decline rate inside the flagged set, by traffic tier:")
display(f.groupby(pd.cut(f["impressions_90d"], [500, 1000, 3000, 10000, 30000, 1e9],
                         labels=["500-1k", "1k-3k", "3k-10k", "10k-30k", "30k+"]), observed=True)["is_declining"]
         .agg(n="size", decline_rate="mean").round(3))

metrics = {
    "assignment": "ML-07 baseline", "data": "data/raw/content_refresh_anonymized.csv (starter slice)",
    "label_for_eval_only": "trend_direction == 'down'", "rows": int(len(q)), "flagged": int(q["flagged"].sum()),
    "rule": {"position_bands": RULE_BANDS, "min_impressions_90d": MIN_IMPRESSIONS, "low_ctr_share_of_band": LOW_CTR_SHARE,
             "band_median_ctr_pct": {k: float(v) for k, v in band_ctr.items()},
             "score": "ctr_gap * log1p(impressions_90d)", "reason_code": "ctr_below_position_norm",
             "action": "rewrite_title_meta"},
    "signal_verdicts": {"staleness": "MIXED", "ctr_vs_position": "CONFIRMED"},
    "base_rate": round(float(base), 4),
    "precision_at_k": {str(r["K"]): r["precision_at_K"] for r in rows},
    "precision_all_flagged": round(float(q.loc[q["flagged"], "is_declining"].mean()), 4),
    "sensitivity_precision_at_k": {c: dict(zip(sens.index, sens[c].tolist())) for c in sens.columns},
    "forbidden_inputs_checked": FORBIDDEN,
}
with open("work/outputs/w04_baseline_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print("wrote work/outputs/w04_baseline_metrics.json")

,gap x log(impressions) [CHOSEN],missed clicks = impressions x (band_ctr - ctr),gap only (impressions as tie-break)
P@10,0.600,0.400,0.600
P@20,0.600,0.450,0.700
P@50,0.640,0.460,0.780
P@100,0.690,0.440,0.750
P@200,0.700,0.525,0.775
P@500,0.708,0.590,0.782


Decline rate inside the flagged set, by traffic tier:


,n,decline_rate
impressions_90d,,
500-1k,712,0.764
1k-3k,1138,0.737
3k-10k,738,0.698
10k-30k,298,0.634
30k+,145,0.476


wrote work/outputs/w04_baseline_metrics.json


**What the numbers say (read with the tables above):**

- The **flag** works: pages it fires on decline far more often than the base rate.
- The **top of the list is the weak part**. The top 10–20 are only a little above the base rate (lift ≈ 1.1),
  and the queue only pulls clearly ahead from around K=100. Reason: inside the flagged set, the biggest pages decline *least*
  (30k+ impressions ≈ coin flip), and any formula that rewards volume pushes exactly those to the top.
- "Missed clicks" (the most business-friendly formula) is worse than random at the top. Good to know before Week 5.
- **This is the bar for Week 5:** the model must beat these P@K numbers on the same slice and label — and it should
  especially beat the top 10–20, which is where this rule is weakest.

## 3. Top-10 review

*For each of the top 10: the action, why it is there, and what would make it wrong.* (Table first, then one line each.)

In [7]:
top = q.head(10)[["rank", "content_id", "client_id", "action", "reason_code", "baseline_score",
                  "impressions_90d", "clicks_90d", "ctr", "avg_position", "band_ctr", "search_volume",
                  "main_intent", "days_with_impressions", "is_declining"]]
display(top)
print("clients in top 10:", top["client_id"].value_counts().to_dict())

,rank,content_id,client_id,action,reason_code,baseline_score,impressions_90d,clicks_90d,ctr,avg_position,band_ctr,search_volume,main_intent,days_with_impressions,is_declining
0,1,content_c8e9d6ab9013,client_19581e27de,rewrite_title_meta,ctr_below_position_norm,12.2486,208678,0,0.00,9.7,0.24,20.0,informational,88,1
1,2,content_453722754fea,client_f369cb89fc,rewrite_title_meta,ctr_below_position_norm,11.3562,140079,16,0.01,7.6,0.24,10.0,informational,88,1
2,3,content_4a6607efcb46,client_6208ef0f77,rewrite_title_meta,ctr_below_position_norm,11.1723,128068,17,0.01,2.2,0.20,0.0,informational,88,0
3,4,content_39881853ef0c,client_f369cb89fc,rewrite_title_meta,ctr_below_position_norm,11.1455,112434,10,0.01,7.2,0.24,170.0,informational,88,1
4,5,content_c84a0ab98e90,client_f369cb89fc,rewrite_title_meta,ctr_below_position_norm,10.7766,223271,70,0.03,7.8,0.24,0.0,informational,88,0
5,6,content_0919dd345d80,client_4e07408562,rewrite_title_meta,ctr_below_position_norm,10.7146,119217,26,0.02,7.0,0.24,40.0,informational,88,1
6,7,content_8451fc6f034d,client_d029fa3a95,rewrite_title_meta,ctr_below_position_norm,10.6370,272144,75,0.03,2.3,0.20,10.0,informational,88,0
7,8,content_d274ac4158ef,client_4e07408562,rewrite_title_meta,ctr_below_position_norm,10.6224,65138,6,0.01,6.8,0.24,40.0,informational,88,0
8,9,content_e5f459e737b7,client_f369cb89fc,rewrite_title_meta,ctr_below_position_norm,10.4838,56363,3,0.01,5.9,0.24,30.0,transactional,88,1
9,10,content_c1fe78bc4e37,client_19581e27de,rewrite_title_meta,ctr_below_position_norm,10.3303,134055,43,0.03,7.5,0.24,70.0,commercial,88,1


clients in top 10: {'client_f369cb89fc': 4, 'client_19581e27de': 2, 'client_4e07408562': 2, 'client_6208ef0f77': 1, 'client_d029fa3a95': 1}


**One line each** (label shown only for honesty after the fact — it was not used to rank):

1. `content_c8e9d6ab9013` — **rewrite_title_meta**. 208,678 impressions, **0 clicks**, avg position 9.7. *Wrong if:* zero clicks on 200k impressions is a tracking or URL-mapping gap, not a weak title — check click data before anyone rewrites anything. (label: down)
2. `content_453722754fea` — **rewrite_title_meta**. 140k impressions, 16 clicks (0.01% vs a 0.24% band norm) at position 7.6. *Wrong if:* the target keyword gets ~10 searches a month, so these impressions come from many loosely related searches the page was never meant to win. (label: down)
3. `content_4a6607efcb46` — **rewrite_title_meta**. 128k impressions, 17 clicks at position **2.2**. *Wrong if:* a top-3 page with ~0% CTR usually means the results page answers the question itself (answer box, AI summary) — a new title will not fix that. (label: not down)
4. `content_39881853ef0c` — **rewrite_title_meta**. 112k impressions, 10 clicks, position 7.2. *Wrong if:* this client owns 4 of the top 10 (#2, #4, #5, #9) — it may be one site-wide problem (template, tracking) to fix once, not four rewrites. (label: down)
5. `content_c84a0ab98e90` — **rewrite_title_meta**. 223k impressions, 70 clicks, position 7.8, no keyword data. *Wrong if:* the 7.8 average hides a long tail of deep rankings; against its real position mix the CTR may be normal. (label: not down)
6. `content_0919dd345d80` — **rewrite_title_meta**. 119k impressions, 26 clicks, position 7.0, informational. *Wrong if:* the searches are quick-fact questions Google answers on the page itself — clicks were never on offer. (label: down)
7. `content_8451fc6f034d` — **rewrite_title_meta**. 272k impressions (biggest in the top 10), 75 clicks, position 2.3. *Wrong if:* it sits in the 30k+ tier, where flagged pages decline only ~48% — low CTR is the normal price of ranking for huge generic searches, and the page may be stable. (label: not down)
8. `content_d274ac4158ef` — **rewrite_title_meta**. 65k impressions, 6 clicks, position 6.8. *Wrong if:* same client as #6 — if that client's click data is incomplete, both picks are data errors. (label: not down)
9. `content_e5f459e737b7` — **rewrite_title_meta**. 56k impressions, 3 clicks, position 5.9, **transactional**. *Wrong if:* buying searches are taken by ads and product listings above the organic results — the fix is a different page type, not a better title. (label: down)
10. `content_c1fe78bc4e37` — **rewrite_title_meta**. 134k impressions, 43 clicks, position 7.5, commercial. *Wrong if:* it shows up for 'best X' comparison searches the page does not actually answer — then the fix is the content, not the title. (label: down)

**Tally:** 6 of 10 are declining (P@10 = 0.60 vs base 0.54). Confidence in this top 10 is **low-to-medium**.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks (I found more than one — that is the point of the review):**

- **#1 is probably a data problem, not a page problem.** Zero clicks on 208k impressions is not believable as a 'bad title'. Any queue built on CTR needs a 'clicks look broken' guard before it reaches a reviewer.
- **#3 and #7 (positions 2.2 and 2.3) are probably results-page effects.** A top-3 page with ~0% CTR is the classic sign that Google answers the search itself. The rule cannot see the results page, so it cannot tell 'bad title' from 'no click was possible'.
- **The biggest blind spot — keyword mismatch:** every top-10 page targets a keyword with ≤170 searches a month, yet gets 56k–272k impressions. These pages are shown for thousands of other searches, and `avg_position` blends them into one number. Comparing their CTR to a 'position 7' norm is comparing against the wrong yardstick. Week-5 idea (not added here — the baseline stays frozen): use the warehouse query table to measure how spread out a page's searches are.
- **Client clumping:** one client holds 4 of the top 10 and about a third of the top 50. Those may be one problem counted four times.
- **Volume trap:** inside the flagged set, decline falls as traffic rises (≈76% at 500–1k impressions vs ≈48% at 30k+). The `log(impressions)` term pushes exactly the weakest group up the list. That is why P@10 is only 0.60.

**Leakage check (asserted in the cell below):** the rule reads only `impressions_90d`, `ctr` and `avg_position`. No `trend_*` column, no last-30 / prev-30 window, no FlyRank product flags. The label touches the queue only in the grading cell. One honest overlap remains: the 90-day totals cover the same period as the label's 60 days, so every number here is an **observed same-window link**, not a forecast.

In [8]:
# Leakage / honesty checks, all asserted so Run All fails loudly if something slips in.
used = set(rule_inputs)
assert used.isdisjoint(FORBIDDEN), "label-derived column in the rule"
assert "is_declining" not in OUT_COLS, "label written into the queue CSV"
print("1. Rule reads only:", sorted(used), "-> none are label sources or last/prev-30-day windows")
print("2. No product flags: the starter slice has no FlyRank flag/eligibility columns; none read.")
print("3. Label used only after ranking (section 2 grading cell). Queue CSV has no label column.")
print("4. Band CTRs computed from ctr + position only (no label):", {k: float(v) for k, v in band_ctr.items()})
# The one honest overlap: impressions_90d and ctr cover the same 90 days that contain the label's 60-day window.
print("5. Known caveat: impressions_90d / ctr share the 90-day snapshot with the label window -> "
      "results are an observed same-window link, not a forecast. Week 5 should test on a forward window.")
print("6. Clients in top 50:", q.head(50)["client_id"].nunique(), "| largest client share of top 50:",
      f"{q.head(50)['client_id'].value_counts(normalize=True).iloc[0]:.0%}")

1. Rule reads only: ['avg_position', 'ctr', 'impressions_90d'] -> none are label sources or last/prev-30-day windows
2. No product flags: the starter slice has no FlyRank flag/eligibility columns; none read.
3. Label used only after ranking (section 2 grading cell). Queue CSV has no label column.
4. Band CTRs computed from ctr + position only (no label): {'1-3': 0.2, '4-10': 0.24, '11-20': 0.17, '21-50': 0.09, '50+': 0.0}
5. Known caveat: impressions_90d / ctr share the 90-day snapshot with the label window -> results are an observed same-window link, not a forecast. Week 5 should test on a forward window.
6. Clients in top 50: 11 | largest client share of top 50: 34%


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed before commit)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous IDs)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with bucket tables and n (both flag-linked): staleness **MIXED**, CTR-vs-position **CONFIRMED**
- [x] One rule: score, one reason code (`ctr_below_position_norm`), action label (`rewrite_title_meta` / `monitor`)
- [x] Queue written from the notebook to `work/outputs/baseline_action_score.csv` (git-ignored by design); receipts in `work/outputs/w04_baseline_metrics.json` (committed)
- [x] No future-window or label-derived inputs (asserted in sections 2 and 4)
- [x] Lane confirmed: Lane 2 — Refresh / Content Opportunity Scoring